In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-ml" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import time

def predictive_performance_estimate(data_estimator, graphs, targets, n_rep=3):
    start = time.time()
    scores = []
    mistakes_per_run = []
    sizes_per_run = []
    for i in range(n_rep):
        train_graphs, test_graphs, train_targets, test_targets = train_test_split(graphs, targets, train_size=.7, random_state=i)
        data_estimator.fit(train_graphs, train_targets)
        preds = data_estimator.predict_proba(test_graphs)
        preds = preds[:,-1]
        score = roc_auc_score(test_targets, preds)
        scores.append(score)
        # Compute number of mistakes using a 0.5 threshold on positive class probability
        y_hat = (preds >= 0.5).astype(int)
        mistakes = int(np.sum(y_hat != np.asarray(test_targets)))
        mistakes_per_run.append(mistakes)
        sizes_per_run.append(len(test_targets))
    mean, std = np.mean(scores), np.std(scores)
    end = time.time()
    elapsed = (end - start)/n_rep
    avg_mistakes = float(np.mean(mistakes_per_run)) if mistakes_per_run else 0.0
    avg_sizes = float(np.mean(sizes_per_run)) if sizes_per_run else 0.0
    return scores, mean, std, elapsed, avg_mistakes, avg_sizes

from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

def get_estimator(graph_vectorizer, estimator_type='linear'):
    if estimator_type == 'linear':
        estimator = LogisticRegression(
            solver="saga",        # robust for large/sparse
            penalty="l2",
            class_weight="balanced",
            max_iter=5000,
            n_jobs=-1,
            random_state=0
        )
    else:
        estimator = ExtraTreesClassifier(
            n_estimators=300, n_jobs=-1, random_state=0
        )
    return GraphEstimator(
        transformer=graph_vectorizer,
        estimator=estimator,
        manifold=None,
        n_selected_features=None,
    )


def make_estimator(df):
    from abstractgraph.vectorize import AbstractGraphTransformer
    graph_vectorizer = AbstractGraphTransformer(
        nbits=14, 
        decomposition_function=df, 
        return_dense=True, 
        n_jobs=-1)
    data_estimator = get_estimator(graph_vectorizer, estimator_type='nlinear')
    return data_estimator

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")
assay_ids = loader.available_assay_ids()
print("available assay ids:", assay_ids)

datasets = []
for assay_id in assay_ids:
    original_graphs, original_targets = loader.load(
        assay_id,
        limit_active=1500,
        limit_inactive=1500,
    )
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

In [ ]:
from abstractgraph.operators import *

decomposition_functions = []

df = compose_product(binary_combination(distance=0), cycle(), compose(neighborhood(radius=(0,2)), tree()))
decomposition_functions.append(('cycle',df))

df = graphlet(radius=3, number_of_nodes=(4,6))
decomposition_functions.append(('graphlet',df))

df = add(neighborhood(radius=(0,2)), compose(combination(number_of_elements=2,distance=(1,4)), neighborhood(radius=(0,2))))
decomposition_functions.append(('pairs',df))

df = neighborhood(radius=(0,2))
decomposition_functions.append(('neighborhood',df))

In [ ]:
%%time

results = []

for i, (assay_id, graphs, targets) in enumerate(datasets):
    print('_'*100)
    print(f'{i+1:3d}/{len(datasets)}  AID{assay_id}   #graphs:{len(graphs)}')
    for name, df in decomposition_functions:
        data_estimator = make_estimator(df)
        scores, mean, std, elapsed, avg_mistakes, avg_sizes = predictive_performance_estimate(
            data_estimator, graphs, targets, n_rep=2
        )
        accuracy = 1 - avg_mistakes / avg_sizes
        print(
            f"{name:15s} ROC-AUC: {mean:.3f} ± {std:.3f}   #errs:{avg_mistakes}/{avg_sizes}   acc:{accuracy:.2f}   {elapsed:5.1f}s  [{elapsed/60:.1f}m]"
        )
        results.append(
            {
                'dataset_idx': i + 1,
                'assay_id': assay_id,
                'df': name,
                'roc_auc': mean,
                'roc_auc_std': std,
                'elapsed': elapsed,
                'accuracy': accuracy,
            }
        )

results_df = pd.DataFrame(results)
results_df.to_csv('res.csv', index=False)

In [ ]:
results_df = pd.read_csv('res.csv', index_col=0)
results_df.head()

In [ ]:
from abstractgraph.utils import plot_dataset_method_bars
_ = plot_dataset_method_bars(
    results_df,
    dataset_col='assay_id',
    dataset_label_col='assay_id',
    method_col='df',
    metric_col='accuracy',
    std_col='accuracy_std',
    ylabel='Accuracy',
    title='Accuracy by dataset and method',
    label_rotation=30,
)


In [ ]:
from abstractgraph.utils import plot_dataset_method_bars
_ = plot_dataset_method_bars(
    results_df,
    dataset_col='assay_id',
    dataset_label_col='assay_id',
    method_col='df',
    metric_col='roc_auc',
    std_col='roc_auc_std',
    ylabel='ROC-AUC',
    title='ROC-AUC by dataset and method',
    label_rotation=30,
)


In [ ]:
from abstractgraph.utils import plot_pareto
_ = plot_pareto(
    results_df,
    dataset_col='assay_id',
    method_col='df',
    axis_1_col='elapsed',
    axis_2_col='roc_auc',
    axis_1_label='Time per DF (s)',
    axis_2_label='ROC-AUC',
    axis_1_ascending=False,
    axis_2_ascending=True,
    title='ROC-AUC vs Time per DF/AID',
)


---